In [3]:
import os
print(os.getcwd())

C:\Users\hp\Desktop\Ecommerce-Sales-Analytics\notebooks


In [4]:
# Import libraries for data handling and machine learning
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load your existing processed data - using ../ since this notebook is inside notebooks/ folder
rfm = pd.read_csv('../data/processed/customer_rfm.csv')
dashboard_orders = pd.read_csv('../data/processed/dashboard_orders.csv')

print(rfm.shape)
rfm.head()

(93358, 9)


,customer_unique_id,recency,frequency,monetary,r_score,f_score,m_score,rfm_score,segment
0,0000366f3b9a7992bf8c76cfdf3221e2,112,1,141.90,4,1,4,414,New Customers
1,0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,27.19,4,1,1,411,New Customers
2,0000f46a3911fa3c0805444483337064,537,1,86.22,1,1,2,112,Lost
3,0000f6ccb0745a6a4b88665a16c9f078,321,1,43.62,2,1,1,211,Lost
4,0004aac84e0df4da2b147fca70cf8255,288,1,196.89,2,1,4,214,Lost


In [5]:
# Look at the distribution of recency to pick a sensible churn threshold
print(rfm['recency'].describe())

count    93358.000000
mean       237.941773
std        152.591453
min          1.000000
25%        114.000000
50%        219.000000
75%        346.000000
max        714.000000
Name: recency, dtype: float64


In [6]:
# Define churn threshold: customers in the top 25% most inactive (highest recency) are labeled churned
# quantile(0.75) finds the value below which 75% of recency values fall
churn_threshold = rfm['recency'].quantile(0.75)
print(f"Churn threshold: {churn_threshold} days")

# Create the churn label: 1 = churned (inactive beyond threshold), 0 = active
rfm['churned'] = (rfm['recency'] > churn_threshold).astype(int)

# Check the split between churned and active customers
print(rfm['churned'].value_counts())
print(rfm['churned'].value_counts(normalize=True))  # shows as percentages

Churn threshold: 346.0 days
churned
0    70047
1    23311
Name: count, dtype: int64
churned
0    0.750305
1    0.249695
Name: proportion, dtype: float64


In [7]:
# First, we need customer_unique_id in the dashboard_orders data to link the two tables
# But dashboard_orders only has customer_id (order-level), so let's check if customer_unique_id exists there
print(dashboard_orders.columns.tolist())

['order_id', 'customer_id', 'customer_state', 'customer_city', 'order_purchase_timestamp', 'order_year_month', 'order_status', 'product_category_name_english', 'seller_id', 'seller_state', 'item_revenue', 'price', 'freight_value', 'delivery_delay_days', 'review_score']


In [8]:
# Load the raw customers table again, just to grab the customer_id -> customer_unique_id mapping
df_customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')

# Merge this mapping into dashboard_orders using customer_id as the key
dashboard_orders = dashboard_orders.merge(
    df_customers[['customer_id', 'customer_unique_id']], 
    on='customer_id', 
    how='left'
)

# Confirm it worked
print(dashboard_orders.columns.tolist())
print(dashboard_orders[['customer_id', 'customer_unique_id']].head())

['order_id', 'customer_id', 'customer_state', 'customer_city', 'order_purchase_timestamp', 'order_year_month', 'order_status', 'product_category_name_english', 'seller_id', 'seller_state', 'item_revenue', 'price', 'freight_value', 'delivery_delay_days', 'review_score', 'customer_unique_id']
                        customer_id                customer_unique_id
0  9ef432eb6251297304e76186b10a928d  7c396fd4830fd04220f754e42b4e5bff
1  9ef432eb6251297304e76186b10a928d  7c396fd4830fd04220f754e42b4e5bff
2  9ef432eb6251297304e76186b10a928d  7c396fd4830fd04220f754e42b4e5bff
3  b0830fb4747a6c6d20dea0b8c802d7ef  af07308b275d755c9edb36a90c618231
4  41ce2a54c0b03bf3443c3d931a367089  3a653a41f6f9fc3d2a113cf8398680e8


In [9]:
# Group by customer_unique_id to get average delivery delay and average review score per customer
customer_features = dashboard_orders.groupby('customer_unique_id').agg(
    avg_delivery_delay=('delivery_delay_days', 'mean'),
    avg_review_score=('review_score', 'mean')
).reset_index()

print(customer_features.shape)
customer_features.head()

(93358, 3)


,customer_unique_id,avg_delivery_delay,avg_review_score
0,0000366f3b9a7992bf8c76cfdf3221e2,-5.0,5.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,-5.0,4.0
2,0000f46a3911fa3c0805444483337064,-2.0,3.0
3,0000f6ccb0745a6a4b88665a16c9f078,-12.0,4.0
4,0004aac84e0df4da2b147fca70cf8255,-8.0,5.0


In [10]:
# Combine RFM + churn label with the new delivery/review features
model_data = rfm.merge(customer_features, on='customer_unique_id', how='left')

# Check for any missing values after the merge (some customers might lack delivery/review data)
print(model_data.isnull().sum())
model_data.head()

customer_unique_id      0
recency                 0
frequency               0
monetary                0
r_score                 0
f_score                 0
m_score                 0
rfm_score               0
segment                 0
churned                 0
avg_delivery_delay      8
avg_review_score      603
dtype: int64


,customer_unique_id,recency,frequency,monetary,r_score,f_score,m_score,rfm_score,segment,churned,avg_delivery_delay,avg_review_score
0,0000366f3b9a7992bf8c76cfdf3221e2,112,1,141.90,4,1,4,414,New Customers,0,-5.0,5.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,27.19,4,1,1,411,New Customers,0,-5.0,4.0
2,0000f46a3911fa3c0805444483337064,537,1,86.22,1,1,2,112,Lost,1,-2.0,3.0
3,0000f6ccb0745a6a4b88665a16c9f078,321,1,43.62,2,1,1,211,Lost,0,-12.0,4.0
4,0004aac84e0df4da2b147fca70cf8255,288,1,196.89,2,1,4,214,Lost,0,-8.0,5.0


In [11]:
# Check how many rows have missing values in our feature columns
print(model_data[['avg_delivery_delay', 'avg_review_score']].isnull().sum())

# Drop rows with missing delivery/review data - customers with no delivered orders won't have this info
# (a small number of rows, safe to drop for modeling purposes)
model_data = model_data.dropna(subset=['avg_delivery_delay', 'avg_review_score'])
print("Rows after dropping missing values:", model_data.shape[0])

avg_delivery_delay      8
avg_review_score      603
dtype: int64
Rows after dropping missing values: 92747


In [12]:
# Select the features the model will learn from
# We use recency, frequency, monetary, delivery delay, and review score as predictors
# We exclude the churn label itself and identifier columns
feature_columns = ['recency', 'frequency', 'monetary', 'avg_delivery_delay', 'avg_review_score']

X = model_data[feature_columns]  # Features (inputs)
y = model_data['churned']         # Target (what we're predicting: 1 = churned, 0 = active)

print(X.shape, y.shape)

(92747, 5) (92747,)


In [13]:
# Split data: 80% to train the model, 20% held back to test how well it generalizes
# random_state=42 makes this split reproducible (same split every time you run it)
# stratify=y ensures both train and test sets have the same churn/active ratio as the full dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set size:", X_train.shape[0])
print("Test set size:", X_test.shape[0])

Training set size: 74197
Test set size: 18550


In [14]:
# Random Forest: builds many decision trees and combines their votes - handles non-linear patterns well
# and gives us feature importance (which factors matter most for predicting churn)
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Fit the model on training data - this is where the actual "learning" happens
model.fit(X_train, y_train)

print("Model trained successfully")

Model trained successfully


In [15]:
# Use the trained model to predict churn on the test set (data it hasn't seen before)
y_pred = model.predict(X_test)

# Accuracy: what % of predictions were correct overall
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2%}")

# Classification report: shows precision, recall, and f1-score for each class (churned vs active)
# Precision = of predicted churners, how many actually churned
# Recall = of actual churners, how many did we catch
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Active', 'Churned']))

Accuracy: 100.00%

Classification Report:
              precision    recall  f1-score   support

      Active       1.00      1.00      1.00     13923
     Churned       1.00      1.00      1.00      4627

    accuracy                           1.00     18550
   macro avg       1.00      1.00      1.00     18550
weighted avg       1.00      1.00      1.00     18550



In [16]:
# Confusion matrix shows: true negatives, false positives, false negatives, true positives
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)
print("\n[Row 0 = Actual Active, Row 1 = Actual Churned]")
print("[Col 0 = Predicted Active, Col 1 = Predicted Churned]")

Confusion Matrix:
[[13923     0]
 [    0  4627]]

[Row 0 = Actual Active, Row 1 = Actual Churned]
[Col 0 = Predicted Active, Col 1 = Predicted Churned]


In [18]:
# Extract which features mattered most to the model's decisions
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance)

              feature  importance
0             recency    0.990980
3  avg_delivery_delay    0.007561
2            monetary    0.000817
4    avg_review_score    0.000570
1           frequency    0.000072


In [19]:
# Remove recency from the feature list since it directly defines the churn label (data leakage)
# The model should predict churn using OTHER signals, not the same rule used to create the label
feature_columns = ['frequency', 'monetary', 'avg_delivery_delay', 'avg_review_score']

X = model_data[feature_columns]
y = model_data['churned']

# Re-split with the corrected features
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Retrain the model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Re-evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2%}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Active', 'Churned']))

Accuracy: 66.87%

Classification Report:
              precision    recall  f1-score   support

      Active       0.77      0.80      0.78     13923
     Churned       0.32      0.29      0.30      4627

    accuracy                           0.67     18550
   macro avg       0.54      0.54      0.54     18550
weighted avg       0.66      0.67      0.66     18550



In [20]:
# Re-check feature importance with the corrected feature set
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(feature_importance)

              feature  importance
1            monetary    0.878992
2  avg_delivery_delay    0.097244
3    avg_review_score    0.020948
0           frequency    0.002815


In [21]:
# class_weight='balanced' tells the model to pay more attention to the minority class (Churned)
# instead of optimizing purely for overall accuracy
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Active', 'Churned']))

Accuracy: 63.78%

Classification Report:
              precision    recall  f1-score   support

      Active       0.78      0.73      0.75     13923
     Churned       0.31      0.37      0.34      4627

    accuracy                           0.64     18550
   macro avg       0.54      0.55      0.54     18550
weighted avg       0.66      0.64      0.65     18550



In [22]:
# Save feature importance and final metrics to processed folder, so you can reference exact numbers later
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

feature_importance.to_csv('../data/processed/churn_feature_importance.csv', index=False)
print(feature_importance)

              feature  importance
1            monetary    0.867634
2  avg_delivery_delay    0.107621
3    avg_review_score    0.021915
0           frequency    0.002830


In [23]:
# Also save the model_data with churn predictions for reference
model_data.to_csv('../data/processed/customer_churn_data.csv', index=False)
print("Saved churn modeling data")

Saved churn modeling data


In [24]:
git add .
git commit -m "Add churn prediction model with RFM and delivery/review features"
git push

SyntaxError: invalid syntax (3286476590.py, line 1)